In [8]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [11]:
!pip install -q streamlit torch torchvision timm opencv-python-headless matplotlib numpy Pillow requests scikit-learn pandas

In [12]:
!pip install -q streamlit torch torchvision timm opencv-python-headless matplotlib numpy Pillow requests scikit-learn pandas

In [9]:
import os, subprocess, time, requests, sys

BASE_DIR = "/content/drive/MyDrive/capcapstone/dr_detection"
APP_DIR = os.path.join(BASE_DIR, "raw_data")

if not os.path.exists(APP_DIR):
    print(f"Directory {APP_DIR} still doesn't exist. Please double-check your Drive files structure.")
else:
    print("Path found! Stopping previous Streamlit processes...")
    subprocess.run(["pkill", "-f", "streamlit"], stderr=subprocess.DEVNULL)
    subprocess.run(["fuser", "-k", "8501/tcp"], stderr=subprocess.DEVNULL)
    time.sleep(2)

    os.chdir(APP_DIR)
    log_path = os.path.join(APP_DIR, "streamlit.log")
    log_file = open(log_path, "w")

    print("Launching Streamlit server...")
    proc = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0",
        "--server.headless", "true", "--server.enableCORS", "false", "--server.enableXsrfProtection", "false",
        "--browser.gatherUsageStats", "false", "--theme.base", "light"], stdout=log_file, stderr=subprocess.STDOUT)

    ready = False
    for i in range(15):
        time.sleep(1)
        if proc.poll() is not None:
            print("Streamlit crashed quickly. Log contents:")
            break
        try:
            if requests.get("http://localhost:8501", timeout=1).status_code == 200:
                ready = True
                print(f"Streamlit is running on port 8501!")
                break
        except requests.exceptions.ConnectionError:
            pass

    log_file.flush()
    print("--- Recent Logs ---")
    print(open(log_path).read()[-2000:])

Path found! Stopping previous Streamlit processes...
Launching Streamlit server...
Streamlit crashed quickly. Log contents:
--- Recent Logs ---
/usr/bin/python3: No module named streamlit



In [20]:
from google.colab import output
# Render the Streamlit UI as an iframe inside this notebook
output.serve_kernel_port_as_iframe(8501, height=1000)

<IPython.core.display.Javascript object>

In [ ]:
!pip install -q streamlit torch torchvision timm opencv-python-headless matplotlib numpy Pillow requests scikit-learn pandas

In [ ]:
import os, shutil
BASE_DIR = "/content/drive/MyDrive/capcapstone/dr_detection"
APP_DIR = os.path.join(BASE_DIR, "raw_data")
APP_PATH = os.path.join(APP_DIR, "app.py")
MODEL_PATH = os.path.join(BASE_DIR, "best_model.pth")
for n, p in {"app.py": APP_PATH, "model.py": os.path.join(APP_DIR, "model.py"), "best_model": MODEL_PATH}.items():
    print("OK     " if os.path.exists(p) else "MISSING", n, p)
    assert os.path.exists(p), f"missing {p}"
if not os.path.exists(APP_PATH + ".bak"):
    shutil.copy(APP_PATH, APP_PATH + ".bak")

OK      app.py /content/drive/MyDrive/capcapstone/dr_detection/raw_data/app.py
OK      model.py /content/drive/MyDrive/capcapstone/dr_detection/raw_data/model.py
OK      best_model /content/drive/MyDrive/capcapstone/dr_detection/best_model.pth


In [ ]:
import sys, torch, inspect, importlib
sys.path.insert(0, APP_DIR)
import model as M; importlib.reload(M)
print("Constructor:", inspect.signature(M.HybridDRModel.__init__))
ck = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)
print("Checkpoint keys:", list(ck.keys())[:10] if isinstance(ck, dict) else type(ck))
for k, v in (ck.items() if isinstance(ck, dict) else []):
    if isinstance(v, (int, float)): print("  ", k, "=", v)

model.safetensors: reconstructing file:   0%|          |  0.00B / 77.9MB            

model.safetensors: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Constructor: (self, num_classes=5, dropout=0.3)
Checkpoint keys: ['eff.conv_stem.weight', 'eff.bn1.weight', 'eff.bn1.bias', 'eff.bn1.running_mean', 'eff.bn1.running_var', 'eff.bn1.num_batches_tracked', 'eff.blocks.0.0.conv_dw.weight', 'eff.blocks.0.0.bn1.weight', 'eff.blocks.0.0.bn1.bias', 'eff.blocks.0.0.bn1.running_mean']


In [25]:
import subprocess, time, requests, os, sys
subprocess.run(["pkill", "-f", "streamlit"], stderr=subprocess.DEVNULL)
subprocess.run(["fuser", "-k", "8501/tcp"], stderr=subprocess.DEVNULL)
time.sleep(2)
os.chdir(APP_DIR)
log_path = os.path.join(APP_DIR, "streamlit.log")
log_file = open(log_path, "w")
proc = subprocess.Popen([sys.executable, "-m", "streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0",
    "--server.headless", "true", "--server.enableCORS", "false", "--server.enableXsrfProtection", "false",
    "--browser.gatherUsageStats", "false", "--theme.base", "light"], stdout=log_file, stderr=subprocess.STDOUT)
ready = False
for i in range(60):
    time.sleep(1)
    if proc.poll() is not None: print("Streamlit crashed."); break
    try:
        if requests.get("http://localhost:8501", timeout=1).status_code == 200:
            ready = True; print(f"Streamlit is up after {i+1}s"); break
    except requests.exceptions.ConnectionError: pass
if not ready:
    log_file.flush(); print(open(log_path).read()[-3000:])

Streamlit is up after 3s


In [21]:
from google.colab import output
output.serve_kernel_port_as_window(8501)   # click the link; if blank, use the iframe below
# output.serve_kernel_port_as_iframe(8501, height=1000)

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [24]:
import os

filepath = os.path.join(APP_DIR, 'app.py')
with open(filepath, 'r') as f:
    content = f.read()

# We will replace any existing model metrics blocks with the genuine metrics from test_metrics.csv
# Let's clean up the previous template and insert the verified results.

# Locate target footer so we place it right before the footer
target_footer = """# ============================================================
# FOOTER
# ============================================================"""

# Remove our previous mock metrics template block if it exists
if "MODEL METRICS & EVALUATION RESULTS" in content:
    # We split and rebuild to keep the file clean from duplicates
    parts = content.split("# --------------------------------------------------------\n    # MODEL METRICS & EVALUATION RESULTS")
    content = parts[0] + "\n" + target_footer

# Write the clean, real metric block template using your authentic files
metrics_ui_code = """
    # --------------------------------------------------------
    # MODEL METRICS & EVALUATION RESULTS
    # --------------------------------------------------------
    st.markdown(\"\"\"
    <div class='glass-card' style='margin-top: 25px;'>
        <h3 style='color: #1e293b; margin-bottom: 15px;'>📊 Real Evaluation Results (Test Set)</h3>
        <p style='color: #64748b;'>These are the actual, verified evaluation metrics extracted directly from <code>test_metrics.csv</code> and <code>test_report.txt</code> for our hybrid <b>EfficientNet-B4 + ViT-Base/16</b> model:</p>
        <table style='width:100%; border-collapse: collapse; margin-top: 15px;'>
            <tr style='background-color: #f1f5f9;'>
                <th style='padding: 10px; text-align: left; border-bottom: 2px solid #cbd5e1;'>Metric</th>
                <th style='padding: 10px; text-align: center; border-bottom: 2px solid #cbd5e1;'>Value</th>
            </tr>
            <tr>
                <td style='padding: 10px; border-bottom: 1px solid #e2e8f0;'>Accuracy (Overall)</td>
                <td style='padding: 10px; text-align: center; font-weight: bold; color: #10b981; border-bottom: 1px solid #e2e8f0;'>69.05%</td>
            </tr>
            <tr>
                <td style='padding: 10px; border-bottom: 1px solid #e2e8f0;'>Weighted F1-Score</td>
                <td style='padding: 10px; text-align: center; font-weight: bold; color: #3b82f6; border-bottom: 1px solid #e2e8f0;'>0.70</td>
            </tr>
            <tr>
                <td style='padding: 10px; border-bottom: 1px solid #e2e8f0;'>Quadratic Weighted Kappa (QWK)</td>
                <td style='padding: 10px; text-align: center; font-weight: bold; color: #8b5cf6; border-bottom: 1px solid #e2e8f0;'>0.8373</td>
            </tr>
            <tr>
                <td style='padding: 10px; border-bottom: 1px solid #e2e8f0;'>Area Under Curve (AUC)</td>
                <td style='padding: 10px; text-align: center; font-weight: bold; color: #f59e0b; border-bottom: 1px solid #e2e8f0;'>0.9009</td>
            </tr>
        </table>
    </div>
    \"\"\", unsafe_allow_html=True)
"""

new_content = content.replace(target_footer, metrics_ui_code + "\n" + target_footer)
with open(filepath, 'w') as f:
    f.write(new_content)

print("Successfully written real evaluation metrics directly into app.py!")

Successfully written real evaluation metrics directly into app.py!


In [23]:
import os

outputs_dir = "/content/drive/MyDrive/capcapstone/dr_detection/outputs"
if os.path.exists(outputs_dir):
    print(f"Files in outputs folder: {os.listdir(outputs_dir)}")
    # Let's read file contents to find accuracy, loss, f1-score, etc.
    for file in os.listdir(outputs_dir):
        file_path = os.path.join(outputs_dir, file)
        if os.path.isfile(file_path) and file.endswith(('.txt', '.json', '.csv', '.log')):
            print(f"\n--- Content of {file} ---")
            with open(file_path, 'r') as f:
                print(f.read()[:2000])
else:
    print(f"The directory {outputs_dir} does not exist. Please check the path.")

Files in outputs folder: ['gradcam_heatmaps', 'shap_plots', 'metrics_log.csv', 'model_comparison.csv', 'training_curves.png', 'test_metrics.csv', 'test_report.txt', 'confusion_matrix.png', 'gradcam', 'gradcam_gallery.png', 'iou_results.csv', 'training_curves_v2.png', 'attention_gallery.png', 'ordinal_metrics_log_lam0.0_seed42.csv', 'ordinal_metrics_log_lam0.25_seed42.csv', 'ordinal_metrics_log_lam1.0_seed42.csv', 'ordinal_metrics_log_lam0.5_seed42.csv', 'ablation_results.csv', 'ordinal_metrics_log_lam0.0_seed1.csv', 'ordinal_metrics_log_lam0.5_seed1.csv', 'ordinal_metrics_log_lam0.0_seed2.csv', 'ordinal_metrics_log_lam0.5_seed2.csv', 'branch_share.csv', 'shap_branch_share.png']

--- Content of metrics_log.csv ---
epoch,train_loss,val_loss,acc,f1,qwk,auc
1,1.5600735968854054,1.5583150779659098,0.43902439024390244,0.21560798548094376,0.24841002618780383,0.672006882006882
1,1.5767493607049965,2.0267801880836487,0.2682926829268293,0.1043859649122807,-0.03271877434432624,0.6347552447552447


In [5]:
import os
BASE_DIR = "/content/drive/MyDrive/capcapstone/dr_detection"
APP_DIR = os.path.join(BASE_DIR, "raw_data")
log_path = os.path.join(APP_DIR, "streamlit.log")

if os.path.exists(log_path):
    print("--- Streamlit Log Output ---")
    print(open(log_path).read()[-4000:])
else:
    print(f"Log file not found at {log_path}. Make sure to run the cell above (p_xH-dOQGntd) first to spin up the server!")

Log file not found at /content/drive/MyDrive/capcapstone/dr_detection/raw_data/streamlit.log. Make sure to run the cell above (p_xH-dOQGntd) first to spin up the server!


In [6]:
import os
if os.path.exists(APP_DIR):
    print(f"Files in {APP_DIR}:")
    print(os.listdir(APP_DIR))
else:
    print(f"APP_DIR does not exist: {APP_DIR}")

APP_DIR does not exist: /content/drive/MyDrive/capcapstone/dr_detection/raw_data


In [7]:
import os
drive_path = "/content/drive/MyDrive"
if os.path.exists(drive_path):
    print("Folders in your Google Drive:")
    print(os.listdir(drive_path))
else:
    print("Google Drive root '/content/drive/MyDrive' is not accessible. Please run cell yQkuR4y4GdBp to mount Google Drive first.")

Google Drive root '/content/drive/MyDrive' is not accessible. Please run cell yQkuR4y4GdBp to mount Google Drive first.


In [4]:
import os
BASE_DIR = "/content/drive/MyDrive/capcapstone/dr_detection"
APP_DIR = os.path.join(BASE_DIR, "raw_data")
log_path = os.path.join(APP_DIR, "streamlit.log")

if os.path.exists(log_path):
    print("--- Streamlit Log (Last 100 lines) ---")
    with open(log_path, "r") as f:
        lines = f.readlines()
        print("".join(lines[-100:]))
else:
    print(f"Log file not found at {log_path}")

Log file not found at /content/drive/MyDrive/capcapstone/dr_detection/raw_data/streamlit.log


In [ ]:
import sys
!{sys.executable} -m pip install -q streamlit

In [ ]:
import os
# Let's search for attention_rollout or custom_forward in the app and model files
for filename in ['app.py', 'model.py']:
    filepath = os.path.join(APP_DIR, filename)
    if os.path.exists(filepath):
        with open(filepath, 'r') as f:
            content = f.read()
            if 'custom_forward' in content or 'attention_rollout' in content:
                print(f"Found target in {filename}!")

Found target in app.py!


In [ ]:
import os
filepath = os.path.join(APP_DIR, 'app.py')
with open(filepath, 'r') as f:
    lines = f.readlines()
for idx, line in enumerate(lines):
    if 'def custom_forward' in line or 'attention_rollout' in line:
        # Print surrounding lines
        start = max(0, idx - 5)
        end = min(len(lines), idx + 20)
        print(f"--- {filepath} (Lines {start+1} to {end}) ---")
        print("".join(lines[start:end]))

--- /content/drive/MyDrive/capcapstone/dr_detection/raw_data/app.py (Lines 522 to 546) ---

# ============================================================
# ATTENTION ROLLOUT
# ============================================================

def attention_rollout(
    model,
    image_tensor,
):

    attention_maps = []

    try:
        block = model.vit.blocks[-1]

    except Exception:
        return None

    original_forward = block.attn.forward

    def custom_forward(x):

        B, N, C = x.shape

        qkv = block.attn.qkv(x)

--- /content/drive/MyDrive/capcapstone/dr_detection/raw_data/app.py (Lines 537 to 561) ---
    except Exception:
        return None

    original_forward = block.attn.forward

    def custom_forward(x):

        B, N, C = x.shape

        qkv = block.attn.qkv(x)

        qkv = qkv.reshape(
            B,
            N,
            3,
            block.attn.num_heads,
            C // block.attn.num_heads,
        )

        qkv = qkv.permute(
           

In [ ]:
import os

filepath = os.path.join(APP_DIR, 'app.py')
with open(filepath, 'r') as f:
    content = f.read()

# Safely replace the signature of custom_forward to accept any extra arguments
old_signature = "def custom_forward(x):"
new_signature = "def custom_forward(x, *args, **kwargs):"

if old_signature in content:
    content = content.replace(old_signature, new_signature)
    with open(filepath, 'w') as f:
        f.write(content)
    print("Successfully patched custom_forward in app.py!")
else:
    print("Could not find the target signature to replace. It might have already been modified.")

Successfully patched custom_forward in app.py!
